In [1]:
!pip install fuzzywuzzy python-levenshtein pandas scikit-learn spacy
!python -m spacy download en_core_web_sm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 2.2 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.5 MB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.9 MB 2.0 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/8.9 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.9 MB 2.4 MB/s eta 0:00:03
   ---------- ----------------------------- 2.4/8.9 MB 2.5 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/8.9 MB 2.5 MB/s eta 0:00:03
   --------------- ------------------------ 3.4/8.9 MB 2.6 MB/s eta 0:00:03
   ----------------- ----------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.9 MB/s eta 0:00:07
     -- ------------------------------------- 0.8/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.3/12.8 MB 1.7 MB/s eta 0:00:07
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:07
     ------ --------------------------------- 2.1/12.8 MB 1.8 MB/s eta 0:00:06
     -------- ------------------------------- 2.6/12.8 MB 1.9 MB/s eta 0:00:06
     --------- ------------------------------ 3.1/12.8 MB 2.0 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 1.9 MB/s eta 0:00:05
     ----------- ---------------------------- 3.7/12.8 MB 2.0 MB/s eta 0:00:05
     ------------- -------------------------- 4.2/12.8 MB 1.9 MB/s eta 0:00:05
     ------------- -------------------------- 4.5/12.8 MB 1.9 MB/s


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from fuzzywuzzy import fuzz, process
import re
from collections import defaultdict
import numpy as np

def preprocess_org_name(name):
    """Limpa e padroniza nomes de organizações"""
    if pd.isna(name):
        return ""
    
    # Converter para minúsculas
    name = name.lower()
    
    # Remover termos comuns que não diferenciam organizações
    stop_words = {'inc', 'llc', 'ltd', 'corp', 'corporation', 'company', 'co', 
                  'gmbh', 'ag', 'sa', 'bv', 'pte', 'limited', 'group', 'holding',
                  'international', 'technologies', 'systems', 'services', 'communications'}
    
    # Remover pontuação e dividir em palavras
    words = re.findall(r'\b\w+\b', name)
    
    # Filtrar stop words e palavras muito curtas
    filtered_words = [word for word in words if word not in stop_words and len(word) > 2]
    
    return ' '.join(sorted(set(filtered_words)))  # Remove duplicatas e ordena

def find_similar_organizations(org_list, threshold=85):
    """Encontra organizações similares automaticamente"""
    processed_names = {org: preprocess_org_name(org) for org in org_list}
    
    clusters = []
    used_orgs = set()
    
    for org1 in org_list:
        if org1 in used_orgs:
            continue
            
        cluster = [org1]
        used_orgs.add(org1)
        
        for org2 in org_list:
            if org2 in used_orgs or org1 == org2:
                continue
                
            # Calcular similaridade
            similarity = fuzz.token_set_ratio(processed_names[org1], processed_names[org2])
            
            if similarity >= threshold:
                cluster.append(org2)
                used_orgs.add(org2)
        
        clusters.append(cluster)
    
    return clusters

def create_auto_mapping(org_list, threshold=85):
    """Cria mapeamento automático baseado em similaridade"""
    clusters = find_similar_organizations(org_list, threshold)
    
    mapping = {}
    for cluster in clusters:
        # Escolher o nome mais "limpo" como principal
        main_name = choose_main_name(cluster)
        mapping[main_name] = cluster
    
    return mapping

def choose_main_name(names):
    """Escolhe o nome principal baseado em critérios heurísticos"""
    scores = {}
    
    for name in names:
        score = 0
        
        # Penalizar nomes muito curtos
        if len(name) < 5:
            score -= 10
            
        # Bonus para nomes sem abreviações
        if not any(abbr in name.lower() for abbr in ['inc', 'ltd', 'gmbh', 'corp']):
            score += 5
            
        # Bonus para nomes que contêm palavras-chave da empresa
        if any(keyword in name.lower() for keyword in ['amazon', 'google', 'microsoft', 'cloud']):
            score += 3
            
        scores[name] = score
    
    return max(scores.items(), key=lambda x: x[1])[0]